# 实验一：AI 驱动的网络流量分类（Network Traffic Classification）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU）&nbsp;|&nbsp; **无需 GPU**

---

## 实验概述

网络流量分类（Network Traffic Classification）是现代网络管理的基础任务：识别每一条网络流属于哪种应用类型，例如 Web 浏览、DNS 查询、视频流媒体还是文件下载。准确的流量分类是带宽管理、QoS（服务质量保障）、安全检测和网络优化的前提。

传统方法依赖**端口号（Port Number）**：HTTP 用 80 端口，DNS 用 53 端口，FTP 用 21 端口。然而，随着 HTTPS 的普及（几乎所有流量都走 443 端口）、端口伪装和应用层多路复用，仅凭端口号已无法可靠区分应用类型。

本实验展示如何用**流统计特征（Flow Statistical Features）**和**机器学习分类器**取代传统端口规则，实现更可靠的流量识别。实验生成四类仿真流量数据，训练 KNN 分类器，并通过对比实验和混淆矩阵直观揭示两种方法的差距。

本实验对应课程中「AI 赋能网络管理」部分的核心内容。

## 学习目标

完成本实验后，你应该能够：

1. 解释为什么端口号无法可靠地区分现代网络中的应用类型；
2. 理解流统计特征（包长、包间隔、字节数）如何反映不同应用的行为模式；
3. 用自己的语言描述 K 近邻算法（K-Nearest Neighbors, KNN）的分类过程；
4. 读懂混淆矩阵（Confusion Matrix），从中分析哪些类别容易相互混淆；
5. 基于实验结果，判断统计特征方法相对于端口规则的优劣，并说明原因。

## 背景与基本原理

### 为什么端口号不够用？

在互联网早期，不同应用使用固定端口：HTTP→80，FTP→21，DNS→53。网络设备只需检查报文头部的目的端口号，就能判断应用类型。

然而，这种方法在现代网络中面临三个根本问题：

- **HTTPS 的普及**：绝大多数 Web 流量、视频流媒体（YouTube、Netflix）都复用 443 端口，使得单靠端口号无法区分"Web 页面"和"视频流"；
- **端口随机化**：P2P 应用、VoIP 等会随机选择端口，规避基于端口的防火墙策略；
- **应用层多路复用**：QUIC 协议、HTTP/3 等新兴协议将多种应用流量封装在同一 UDP/443 连接中。

### 流统计特征（Flow Statistical Features）

不同应用在行为上存在显著差异，这些差异可以通过**一条 Flow（流）的统计特征**来捕捉：

| 特征 | 含义 | 直觉解释 |
|------|------|----------|
| `pkt_len_mean` | 平均包长（字节） | DNS 请求极小（~60 B），视频/下载接近最大帧（~1400 B） |
| `pkt_len_var` | 包长方差 | 流媒体包长较稳定；Web 页面包长波动大 |
| `gap_mean` | 平均包间隔（秒） | 下载连续发包间隔极短；DNS 查询间隔较长 |
| `gap_var` | 包间隔方差 | 视频恒定码率播放时包间隔方差小 |
| `bytes` | 总字节数 | 下载流字节数远大于 DNS 查询 |

这些特征共同构成了每条 Flow 在**特征空间（Feature Space）**中的位置。不同应用类型在特征空间中形成相对聚集的区域，这正是机器学习分类的基础。

### K 近邻算法（K-Nearest Neighbors, KNN）

KNN 是一种直觉上最易理解的分类算法：

> 对于一条待分类的 Flow，在训练集中找到距它最近的 K 条 Flow，用这 K 条 Flow 中占多数的类别作为预测结果。

**关键概念：**
- **距离**：通常采用欧氏距离（Euclidean Distance）。特征数值差异越大，距离越远。
- **K 值**：本实验使用 K=5，即投票的"邻居数"为 5。K 太小容易过拟合，K 太大可能引入远距离干扰点。
- **无显式训练**：KNN 不需要参数拟合，预测时直接在训练集上搜索最近邻。

注意：特征之间量纲不同（字节 vs 秒），在实际工程中通常需要做特征标准化（Normalization），本实验仿真数据量纲已调整为相近范围，以便直接对比。

## 实验设计

**数据来源**：本实验使用程序生成的仿真（Synthetic）流量特征数据，不依赖真实 pcap 包。每类流量 200 条记录，共 800 条，4 类：Web / DNS / Video / Download。

**为什么用仿真数据**：真实流量数据涉及隐私，且抓包、提取特征需要额外工具。仿真数据保留了真实流量的统计规律，足以演示分类原理。课程完整版实验使用真实 pcapng 文件。

**对比设计**：
- **Baseline**：仅用端口号（1 维特征） + KNN —— 模拟传统规则方法；
- **实验组**：5 维统计特征 + KNN —— 模拟 ML 方法；
- **评价指标**：分类准确率（Accuracy）和混淆矩阵（Confusion Matrix）。

**预期观察**：统计特征的准确率应显著高于端口号 Baseline，且混淆矩阵的对角线数值远大于非对角线。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | NumPy、Pandas、scikit-learn、seaborn、Matplotlib（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
# 环境准备（Kaggle 已预装 scikit-learn，只需确认版本）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("所有依赖已就绪 ✅")

## 步骤一：生成仿真流量数据并观察特征分布

下面的代码做两件事：

**① 生成仿真数据**：用 `generate_flow()` 函数按照每类流量的典型统计特性（参见下表）生成 200 条记录，总共 800 条。

| 流量类型 | 平均包长 | 平均包间隔 | 典型端口 | 行为特点 |
|---------|---------|-----------|---------|----------|
| Web | ~800 B | ~50 ms | 80 | 中等包长，间歇发送 |
| DNS | ~100 B | ~500 ms | 53 | 极小包，低频查询 |
| Video | ~1400 B | ~10 ms | 443 | 大包，高频持续发送 |
| Download | ~1500 B | ~1 ms | 21 | 满包，极密集发送 |

**② 特征空间散点图（左图）**：将 800 条 Flow 投影到「平均包长 × 平均包间隔」二维平面。**观察：不同颜色是否形成了相对分离的聚类？** 这种分离是 KNN 能正确分类的基础。

**③ 准确率对比柱状图（右图）**：同时运行两个 KNN 分类器——一个只用端口号，一个用 5 维统计特征——直接对比准确率差距。

> 思考：端口号 Baseline 的准确率上限在哪里？（提示：想想 4 类流量的端口号分布有多少重叠）

In [ ]:
# 生成演示数据（模拟真实流量特征）
# 4 类流量：Web(0) / DNS(1) / Video(2) / Download(3)
# 特征：平均包长、包长方差、包间隔均值、包间隔方差、字节数、端口号

np.random.seed(42)
n_samples = 200

def generate_flow(label, n, pkt_mean, pkt_var, gap_mean, gap_var, port):
    return pd.DataFrame({
        'pkt_len_mean': np.random.normal(pkt_mean, pkt_mean*0.15, n),
        'pkt_len_var': np.random.normal(pkt_var, pkt_var*0.3, n),
        'gap_mean': np.random.normal(gap_mean, gap_mean*0.2, n),
        'gap_var': np.random.normal(gap_var, gap_var*0.3, n),
        'bytes': np.random.normal(pkt_mean * 20, pkt_mean * 5, n),
        'port': np.random.normal(port, 5, n).astype(int),
        'label': label
    })

df = pd.concat([
    generate_flow(0, n_samples, 800, 50000, 0.05, 0.02, 80),    # Web: 小包、80端口
    generate_flow(1, n_samples, 100, 500, 0.5, 0.1, 53),        # DNS: 极小包、53端口
    generate_flow(2, n_samples, 1400, 100000, 0.01, 0.005, 443), # Video: 大包、443端口
    generate_flow(3, n_samples, 1500, 50000, 0.001, 0.001, 21),  # Download: 满包、21端口
])

labels = ['Web', 'DNS', 'Video', 'Download']
print(f"生成 {len(df)} 条流量记录")
df.head()

In [ ]:
# 可视化：不同类型的流量在特征空间中的分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, label in enumerate(labels):
    subset = df[df['label'] == i]
    axes[0].scatter(subset['pkt_len_mean'], subset['gap_mean'], 
                   label=label, alpha=0.6, s=20)

axes[0].set_xlabel('Avg Packet Length (bytes)')
axes[0].set_ylabel('Avg Inter-arrival Time (s)')
axes[0].set_title('Traffic Feature Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 只用端口号分类 vs 用统计特征分类 —— 直观对比
X_port = df[['port']]
X_stats = df[['pkt_len_mean', 'pkt_len_var', 'gap_mean', 'gap_var', 'bytes']]
y = df['label']

X_train_p, X_test_p, y_train, y_test = train_test_split(X_port, y, test_size=0.3, random_state=42)
X_train_s, X_test_s, _, _ = train_test_split(X_stats, y, test_size=0.3, random_state=42)

knn_port = KNeighborsClassifier(n_neighbors=5).fit(X_train_p, y_train)
knn_stats = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)

acc_port = knn_port.score(X_test_p, y_test)
acc_stats = knn_stats.score(X_test_s, y_test)

axes[1].bar(['仅用端口号', '统计特征（5维）'], [acc_port, acc_stats], 
           color=['#e74c3c', '#2ecc71'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Classification Accuracy')
axes[1].set_title('Impact of Feature Selection')
for i, v in enumerate([acc_port, acc_stats]):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"\n仅用端口号：{acc_port:.1%}  |  统计特征：{acc_stats:.1%}")
print("为什么统计特征远优于端口号？因为同一端口可能承载多种流量类型（如 443 可能是 Web 也可能是视频）")

## 步骤二：用混淆矩阵深入分析分类结果

准确率（Accuracy）只能告诉你「整体对了多少」，却无法揭示**哪些类别最容易相互混淆**。混淆矩阵（Confusion Matrix）提供了更细粒度的信息。

**如何读混淆矩阵：**

```
              预测类别
            Web  DNS  Video  Download
真  Web    [ TP  ...   ...    ...  ]
实  DNS    [ ...  TP   ...    ...  ]
类  Video  [ ...  ...   TP    ...  ]
别  Download[...  ...   ...    TP  ]
```

- **对角线（diagonal）**：预测正确的样本数。数值越大越好。
- **非对角线**：预测错误的样本数。第 i 行第 j 列表示「真实是类别 i，但被误判为类别 j」的样本数。
- **横行**：代表某个真实类别的所有预测分布，行内非对角线数字说明该类被误判为哪些类。
- **纵列**：代表某个预测类别的来源，列内非对角线数字说明该预测类别中混入了哪些其他类别。

**运行下方代码后，请观察**：
- 哪个类别的行中非对角线数字最多？（该类最容易被误判）
- 哪两个类别之间的互混最严重？（统计特征最相似的两类）
- 结合上表中的特征参数，你能解释为什么这两类容易混淆吗？

In [ ]:
# 混淆矩阵：看看哪两类最容易混淆
y_pred = knn_stats.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('True Label')
plt.title('Confusion Matrix: KNN + Statistical Features')
plt.show()

print("\n课堂互动问题：")
print("1. 对角线上的数字代表什么？")
print("2. 哪两个类别最容易混淆？为什么？")
print("3. 如果让你加一个新特征来改善，你会加什么？")

## 实验结果与分析

### 准确率对比

运行实验后，你会看到两个准确率数值：

- **仅端口号（Port-based Baseline）**：准确率通常在 40%–70% 之间，接近随机猜测水平（4 类随机猜测为 25%）。这是因为在仿真数据中，端口号分布存在噪声（用 `np.random.normal` 在目标端口附近随机取整），且不同类别端口号本身区分性有限。

- **统计特征 KNN（ML Approach）**：准确率通常在 90%–99% 之间，远高于端口号 Baseline。

**结论**：即使是简单的 KNN 分类器，只要特征选择得当，也能远超基于固定规则的 Baseline。这正是机器学习方法在网络流量分类中取代传统规则的核心原因。

### 混淆矩阵分析

从混淆矩阵中，通常可以观察到：

- **Web 和 Video**：偶有互混。两者都使用 HTTPS（443 端口区域），包长也相对较大，但 Video 的包间隔更小、更稳定，KNN 依靠 `gap_mean` 特征加以区分。
- **DNS**：几乎不会被误判。DNS 的极小包长（~100 B）和较大包间隔（~500 ms）在特征空间中高度孤立，最容易识别。
- **Download**：极少误判。满载包（~1500 B）加上极短包间隔（~1 ms）是唯一具有这两个极端值的类别。

### 实验局限性

需要特别注意：**本实验使用的是仿真（Synthetic）数据，不代表真实网络环境中的最终性能。** 真实场景中：

1. 同一端口（尤其是 443）可能承载数十种不同应用，特征分布重叠更严重；
2. 加密流量会使某些特征不可观测；
3. 不同网络环境（移动网络、企业内网）的特征分布差异很大，分类器需要重新训练。

## 从实验到实际系统

本实验演示了核心原理，但真实网络流量分类系统面临更多挑战：

- **加密流量**：TLS/HTTPS 加密后，应用层内容无法直接检测，但流统计特征（包长分布、时序特征）仍然可用。近年研究表明，即使是完全加密的流量，也可以通过包序列的统计特征区分应用类型。
- **零日应用（Zero-day Apps）**：训练集中未见过的新应用无法被正确分类，这要求分类器具备开放集识别（Open-set Recognition）能力。
- **特征提取代价**：流统计特征需要在流级别聚合多个包才能计算，存在延迟，不适合需要逐包实时判断的场景。
- **实际使用的模型**：工业界常用随机森林（Random Forest）、XGBoost、或深度学习模型（如 1D-CNN、Transformer）处理更复杂的特征和更大规模数据。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **端口号分类已不可靠**：HTTPS 普及、端口随机化等原因使得传统方法在现代网络中失效。
2. **流统计特征具有分类能力**：包长、包间隔等特征反映了不同应用的行为规律，能有效区分流量类型。
3. **KNN 是简单有效的入门分类器**：通过特征空间中的距离度量，无需复杂训练即可实现分类。
4. **混淆矩阵揭示分类器的薄弱点**：不能只看整体准确率，还要看哪些类别容易混淆。
5. **实验结果依赖特征质量**：特征设计（Feature Engineering）是 ML 性能的关键，而非只有模型本身。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **修改 K 值**：将 `KNeighborsClassifier(n_neighbors=5)` 中的 5 改为 1、10、20，观察准确率如何变化。K 过小和过大各有什么问题？
2. **换用决策树**：代码中已导入 `DecisionTreeClassifier`，用它替换 KNN 并对比结果。决策树的分类结果是否更容易解释？
3. **增加噪声**：在 `generate_flow()` 中将各类特征的标准差（如 `pkt_mean*0.15`）改大，模拟更嘈杂的网络环境，观察准确率如何下降。
4. **增加新特征**：在 `generate_flow()` 中加入「每流包数（packet count）」特征，观察是否提升分类准确率。
5. **加密流量场景**：如果去掉端口号特征，只保留包长和包间隔特征，准确率会下降多少？这模拟了加密流量中无法观测端口的场景。

---

🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验二：自适应视频流与 QoE 优化 →](https://www.kaggle.com/code/guopingtan/fmi-demo2-qoe-optimization)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University